# W6D4: ChromaDB, Similarity Search & RAG Pipeline

## Objectives
- Set up a ChromaDB collection and store 20 documents with embeddings.
- Perform cosine similarity search and metadata filtering.
- Verify retrieved documents and their relevance.
- Load a PDF, split its text into chunks, and embed the chunks in ChromaDB.
- Retrieve the top 3 relevant chunks and pass them to an Ollama LLM to generate an answer.

**Environment:** Python 3.12.10  
**Tools:** ChromaDB, LangChain, Ollama, PyPDF

In [1]:
import chromadb
from chromadb.utils import embedding_functions
from pathlib import Path
from pypdf import PdfReader

print("ChromaDB version:", chromadb.__version__)
print("PDF reader imported successfully")
print("Imports completed successfully!")

ChromaDB version: 1.5.9
PDF reader imported successfully
Imports completed successfully!


In [3]:
# Create a local ChromaDB client
client = chromadb.PersistentClient(path="./chroma_w6d4")

# Create a collection using cosine distance
collection = client.get_or_create_collection(
    name="robotics_knowledge",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_functions.DefaultEmbeddingFunction()
)

print("Collection created:", collection.name)
print("Distance metric: cosine")
print("Initial document count:", collection.count())

Collection created: robotics_knowledge
Distance metric: cosine
Initial document count: 0


In [4]:
documents = [
    "Robotics combines mechanical engineering, electronics, control systems, and computer science.",
    "A robot uses sensors to collect information about its environment.",
    "Actuators convert electrical energy into mechanical motion.",
    "ROS is a framework for developing robot software and connecting robot components.",
    "Computer vision enables robots to interpret images and video.",
    "LiDAR measures distance using laser light and is used for mapping.",
    "Ultrasonic sensors estimate distance using sound waves.",
    "An encoder measures the rotation or position of a motor shaft.",
    "PID control uses proportional, integral, and derivative terms to regulate a system.",
    "SLAM allows a robot to build a map while estimating its own position.",
    "Path planning determines a route from a starting point to a goal.",
    "Obstacle avoidance helps a robot move around objects safely.",
    "A differential-drive robot steers by controlling its left and right wheels.",
    "An IMU measures angular velocity and linear acceleration.",
    "Embedded systems combine hardware and software for dedicated functions.",
    "Machine learning models learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "Object detection identifies objects and their locations in images.",
    "A robotic gripper is used to grasp and manipulate objects.",
    "Battery capacity and power consumption affect a mobile robot's operating time."
]

metadatas = [
    {"topic": "fundamentals", "source": "robotics_notes"},
    {"topic": "sensors", "source": "robotics_notes"},
    {"topic": "actuators", "source": "robotics_notes"},
    {"topic": "software", "source": "robotics_notes"},
    {"topic": "computer_vision", "source": "robotics_notes"},
    {"topic": "sensors", "source": "robotics_notes"},
    {"topic": "sensors", "source": "robotics_notes"},
    {"topic": "sensors", "source": "robotics_notes"},
    {"topic": "control", "source": "robotics_notes"},
    {"topic": "localization", "source": "robotics_notes"},
    {"topic": "planning", "source": "robotics_notes"},
    {"topic": "navigation", "source": "robotics_notes"},
    {"topic": "mobility", "source": "robotics_notes"},
    {"topic": "sensors", "source": "robotics_notes"},
    {"topic": "embedded_systems", "source": "robotics_notes"},
    {"topic": "machine_learning", "source": "robotics_notes"},
    {"topic": "deep_learning", "source": "robotics_notes"},
    {"topic": "computer_vision", "source": "robotics_notes"},
    {"topic": "manipulation", "source": "robotics_notes"},
    {"topic": "power", "source": "robotics_notes"},
]

ids = [f"robotics-{i:02d}" for i in range(1, 21)]

print("Documents prepared:", len(documents))
print("Metadata records:", len(metadatas))
print("Unique IDs:", len(set(ids)))

Documents prepared: 20
Metadata records: 20
Unique IDs: 20


In [5]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)

print("Documents inserted successfully!")
print("Total documents in collection:", collection.count())

Documents inserted successfully!
Total documents in collection: 20


In [6]:
query = "How can a robot determine its position and navigate?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

print("Query:", query)
print("\nTop 3 matching documents:\n")

for i, doc in enumerate(results["documents"][0], start=1):
    metadata = results["metadatas"][0][i - 1]
    distance = results["distances"][0][i - 1]

    print(f"{i}. {doc}")
    print(f"   Topic: {metadata['topic']}")
    print(f"   Cosine distance: {distance:.4f}\n")

Query: How can a robot determine its position and navigate?

Top 3 matching documents:

1. SLAM allows a robot to build a map while estimating its own position.
   Topic: localization
   Cosine distance: 0.3639

2. A robot uses sensors to collect information about its environment.
   Topic: sensors
   Cosine distance: 0.3849

3. Computer vision enables robots to interpret images and video.
   Topic: computer_vision
   Cosine distance: 0.4470



In [7]:
filtered_results = collection.query(
    query_texts=["How do robots measure their surroundings?"],
    n_results=5,
    where={"topic": "sensors"}
)

print("Metadata filter: topic = sensors")
print("Matching documents:", len(filtered_results["documents"][0]))

for i, doc in enumerate(filtered_results["documents"][0], start=1):
    print(f"{i}. {doc}")

Metadata filter: topic = sensors
Matching documents: 5
1. A robot uses sensors to collect information about its environment.
2. Ultrasonic sensors estimate distance using sound waves.
3. LiDAR measures distance using laser light and is used for mapping.
4. An encoder measures the rotation or position of a motor shaft.
5. An IMU measures angular velocity and linear acceleration.


In [8]:
stored_data = collection.get(
    limit=20,
    include=["documents", "metadatas"]
)

print("Total stored documents:", len(stored_data["ids"]))
print("Unique IDs:", len(set(stored_data["ids"])))
print("All documents have metadata:", all(stored_data["metadatas"]))
print("\nFirst stored record:")
print("ID:", stored_data["ids"][0])
print("Document:", stored_data["documents"][0])
print("Metadata:", stored_data["metadatas"][0])

Total stored documents: 20
Unique IDs: 20
All documents have metadata: True

First stored record:
ID: robotics-01
Document: Robotics combines mechanical engineering, electronics, control systems, and computer science.
Metadata: {'source': 'robotics_notes', 'topic': 'fundamentals'}


In [9]:
pdf_files = list(Path(".").glob("*.pdf")) + list(Path("data").glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for pdf in pdf_files:
    print(pdf.resolve())
    

PDF files found: 0


In [10]:
%pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 6.1 MB/s eta 0:00:01
   -------------------------------- ------- 1.6/2.0 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 4.3 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from pathlib import Path
from xml.sax.saxutils import escape

pdf_path = Path("robotics_reference.pdf")

robotics_notes = [
    ("Robotics Overview", "Robotics combines mechanical engineering, electronics, control systems, and computer science. Robots use sensors to perceive their surroundings and actuators to produce movement."),
    ("Sensors and Perception", "LiDAR measures distance using laser light and helps robots build maps. Ultrasonic sensors estimate distance using sound waves. Cameras provide image data for computer vision. An IMU measures angular velocity and linear acceleration, while wheel encoders measure wheel rotation."),
    ("Localization and Navigation", "SLAM allows a robot to build a map while estimating its own position. Path planning determines a route from a starting point to a goal. Obstacle avoidance helps a mobile robot navigate around objects."),
    ("Robot Control", "A PID controller uses proportional, integral, and derivative terms to regulate a system. The proportional term responds to current error, the integral term accumulates past error, and the derivative term responds to the rate of change of error."),
    ("ROS and Robot Software", "Robot Operating System (ROS) provides software libraries and communication tools for robot applications. Nodes perform tasks, topics carry messages between nodes, and services support request-and-response communication."),
    ("Power and Safety", "Battery capacity and power consumption affect a mobile robot's operating time. Robots should use suitable sensors, collision detection, and safe motion limits around people. Sensor readings may contain noise and should be interpreted carefully.")
]

styles = getSampleStyleSheet()
story = []

for heading, content in robotics_notes:
    story.append(Paragraph(escape(heading), styles["Heading2"]))
    story.append(Paragraph(escape(content), styles["BodyText"]))
    story.append(Spacer(1, 12))

doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
doc.build(story)

print("PDF created:", pdf_path.resolve())
print("File exists:", pdf_path.exists())
print("File size (bytes):", pdf_path.stat().st_size)

PDF created: C:\Users\ughan\CynarisInternship\AIML_PROJECT\notebooks\robotics_reference.pdf
File exists: True
File size (bytes): 2689


In [13]:
from pypdf import PdfReader

pdf_path = Path("robotics_reference.pdf")
reader = PdfReader(str(pdf_path))

pdf_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

print("PDF pages:", len(reader.pages))
print("Extracted characters:", len(pdf_text))
print("\nExtracted text preview:\n")
print(pdf_text[:1000])

assert pdf_text.strip(), "No text was extracted from the PDF."

PDF pages: 1
Extracted characters: 1491

Extracted text preview:

Robotics Overview
Robotics combines mechanical engineering, electronics, control systems, and computer science.
Robots use sensors to perceive their surroundings and actuators to produce movement.
Sensors and Perception
LiDAR measures distance using laser light and helps robots build maps. Ultrasonic sensors
estimate distance using sound waves. Cameras provide image data for computer vision. An IMU
measures angular velocity and linear acceleration, while wheel encoders measure wheel rotation.
Localization and Navigation
SLAM allows a robot to build a map while estimating its own position. Path planning determines a
route from a starting point to a goal. Obstacle avoidance helps a mobile robot navigate around
objects.
Robot Control
A PID controller uses proportional, integral, and derivative terms to regulate a system. The
proportional term responds to current error, the integral term accumulates past error, and the
deriv

In [14]:
chunk_size = 450
overlap = 80

chunks = []

start = 0

while start < len(pdf_text):
    end = min(start + chunk_size, len(pdf_text))
    chunk = pdf_text[start:end].strip()

    if chunk:
        chunks.append(chunk)

    if end == len(pdf_text):
        break

    start = end - overlap

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ({len(chunk)} characters) ---")
    print(chunk[:200])

Total chunks created: 4

--- Chunk 1 (450 characters) ---
Robotics Overview
Robotics combines mechanical engineering, electronics, control systems, and computer science.
Robots use sensors to perceive their surroundings and actuators to produce movement.
Sen

--- Chunk 2 (449 characters) ---
ata for computer vision. An IMU
measures angular velocity and linear acceleration, while wheel encoders measure wheel rotation.
Localization and Navigation
SLAM allows a robot to build a map while est

--- Chunk 3 (449 characters) ---
A PID controller uses proportional, integral, and derivative terms to regulate a system. The
proportional term responds to current error, the integral term accumulates past error, and the
derivative t

--- Chunk 4 (380 characters) ---
ns. Nodes perform tasks, topics carry messages between nodes, and services support
request-and-response communication.
Power and Safety
Battery capacity and power consumption affect a mobile robot's o


In [15]:
pdf_collection = client.get_or_create_collection(
    name="robotics_pdf_knowledge",
    metadata={"hnsw:space": "cosine"},
    embedding_function=embedding_functions.DefaultEmbeddingFunction()
)

pdf_ids = [f"pdf-chunk-{i:02d}" for i in range(1, len(chunks) + 1)]

pdf_metadatas = [
    {
        "source": "robotics_reference.pdf",
        "chunk_number": i
    }
    for i in range(1, len(chunks) + 1)
]

# Make rerunning this cell safe
if pdf_collection.count() > 0:
    pdf_collection.delete(ids=pdf_ids[:pdf_collection.count()])

pdf_collection.upsert(
    ids=pdf_ids,
    documents=chunks,
    metadatas=pdf_metadatas
)

print("PDF chunks stored:", pdf_collection.count())
print("Embedding function:", type(pdf_collection._embedding_function).__name__)

PDF chunks stored: 4
Embedding function: DefaultEmbeddingFunction


In [16]:
question = "How does SLAM help a mobile robot navigate?"

retrieved = pdf_collection.query(
    query_texts=[question],
    n_results=3
)

print("Question:", question)
print("\nTop 3 retrieved chunks:\n")

for i, chunk in enumerate(retrieved["documents"][0], start=1):
    metadata = retrieved["metadatas"][0][i - 1]
    distance = retrieved["distances"][0][i - 1]

    print(f"--- Result {i} ---")
    print("Chunk number:", metadata["chunk_number"])
    print("Cosine distance:", round(distance, 4))

Question: How does SLAM help a mobile robot navigate?

Top 3 retrieved chunks:

--- Result 1 ---
Chunk number: 2
Cosine distance: 0.4721
--- Result 2 ---
Chunk number: 1
Cosine distance: 0.4831
--- Result 3 ---
Chunk number: 4
Cosine distance: 0.5968


In [17]:
for i, chunk in enumerate(retrieved["documents"][0], start=1):
    print(f"\n--- Retrieved chunk {i} ---")
    print(repr(chunk))


--- Retrieved chunk 1 ---
'ata for computer vision. An IMU\nmeasures angular velocity and linear acceleration, while wheel encoders measure wheel rotation.\nLocalization and Navigation\nSLAM allows a robot to build a map while estimating its own position. Path planning determines a\nroute from a starting point to a goal. Obstacle avoidance helps a mobile robot navigate around\nobjects.\nRobot Control\nA PID controller uses proportional, integral, and derivative terms to regulate'

--- Retrieved chunk 2 ---
'Robotics Overview\nRobotics combines mechanical engineering, electronics, control systems, and computer science.\nRobots use sensors to perceive their surroundings and actuators to produce movement.\nSensors and Perception\nLiDAR measures distance using laser light and helps robots build maps. Ultrasonic sensors\nestimate distance using sound waves. Cameras provide image data for computer vision. An IMU\nmeasures angular velocity and linear acceleratio'

--- Retrieved chunk 3 ---
"

In [18]:
from langchain_ollama import ChatOllama

context = "\n\n".join(retrieved["documents"][0])

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0
)

prompt = f"""
Answer the question using only the context below.
If the context does not contain enough information, say so.
Do not invent facts.

Context:
{context}

Question:
{question}

Answer:
"""

response = llm.invoke(prompt)

print("Question:", question)
print("\nRAG Answer:\n")
print(response.content)

Question: How does SLAM help a mobile robot navigate?

RAG Answer:

SLAM (Simultaneous Localization and Mapping) helps a mobile robot navigate by allowing it to build a map of its environment while simultaneously estimating its own position within that map. This means the robot can create a detailed map of the areas it explores and at the same time determine its precise location on that map. This capability is crucial for the robot to understand its surroundings and plan efficient paths to reach its goals, even in environments it has not previously encountered.


In [19]:
answer = response.content

slam_supported = (
    "build a map" in answer.lower()
    and "position" in answer.lower()
    and any(
        "SLAM" in chunk and "estimating its own position" in chunk
        for chunk in retrieved["documents"][0]
    )
)

print("Answer mentions map and position:", 
      "build a map" in answer.lower() and "position" in answer.lower())
print("Retrieved context supports the core explanation:", slam_supported)

assert slam_supported, "Review the answer against the retrieved context."
print("RAG verification passed!")

Answer mentions map and position: True
Retrieved context supports the core explanation: True
RAG verification passed!
